# Data Preparation - Patient Medications
This notebook extracts patient medication data from SQL Server CDWWork database and creates a unified dataset combining outpatient prescriptions (RxOut) and inpatient medication administrations (BCMA).

**Source**: SQL Server CDWWork database (RxOut + BCMA schemas)  
**Destination**: `med-data/v1_raw/medications/medications_combined.parquet`

In [ ]:
# Import dependencies

import os
import sys
import logging
import time
from datetime import datetime, timedelta
import pyodbc
import boto3
import pandas as pd
import s3fs
import pyarrow as pa
import pyarrow.parquet as pq
from dotenv import load_dotenv
from importlib.metadata import version
from config import *

In [ ]:
# Verify that dependencies are available for use

def print_version():
    print("boto3:", boto3.__version__)
    print("pandas:", pd.__version__)
    print("s3fs:", s3fs.__version__)
    print("pyarrow:", pa.__version__)
    print("pyodbc:", pyodbc.version)
    print("dotenv:", version("python-dotenv"))


print_version()

In [ ]:
# Set up logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

# Test logging
logging.info("Logging configured successfully")

In [ ]:
# Load configuration from config module

logging.info(f"Configuration loaded: SQL Server={SQLSERVER_SERVER}/{SQLSERVER_DATABASE}")
logging.info(f"MinIO endpoint: {MINIO_ENDPOINT}")
logging.info(f"Destination: s3://{DEST_BUCKET}/{V1_RAW_MEDICATIONS_PREFIX}")
logging.info(f"Default date range: {DEFAULT_START_DATE} to {DEFAULT_END_DATE}")
logging.info(f"BCMA action types: {BCMA_INCLUDE_ACTION_TYPES}")

In [ ]:
# Configure date range for medication extraction
# Modify these variables to adjust the date range filter

# Option 1: Use default (last 365 days)
start_date = DEFAULT_START_DATE
end_date = DEFAULT_END_DATE

# Option 2: Override with custom dates (uncomment to use)
# start_date = datetime(2024, 1, 1).date()
# end_date = datetime(2024, 12, 31).date()

logging.info(f"Using date range: {start_date} to {end_date}")
logging.info(f"Date range span: {(end_date - start_date).days} days")

In [ ]:
# Create SQL Server connection

def create_sqlserver_connection():
    """
    Factory function to create SQL Server connection using pyodbc.
    Returns connection object for CDWWork database.
    """
    logging.info(f"Creating SQL Server connection to {SQLSERVER_SERVER}/{SQLSERVER_DATABASE}")
    
    conn_string = (
        f"DRIVER={{{SQLSERVER_DRIVER}}};"
        f"SERVER={SQLSERVER_SERVER};"
        f"DATABASE={SQLSERVER_DATABASE};"
        f"UID={SQLSERVER_USER};"
        f"PWD={SQLSERVER_PASSWORD};"
        f"TrustServerCertificate={SQLSERVER_TRUST_CERT};"
    )
    
    return pyodbc.connect(conn_string)


# Create the connection
conn = create_sqlserver_connection()
logging.info("SQL Server connection created successfully")
logging.info(f"Connection type: {type(conn)}")

In [ ]:
# Define SQL query for unified medication dataset

# Format action types for SQL IN clause
action_types_sql = "', '".join(BCMA_INCLUDE_ACTION_TYPES)

sql_query = f"""
-- Unified medication dataset: Outpatient + Inpatient
WITH OutpatientMeds AS (
    SELECT 
        p.PatientSID,
        p.PatientIEN,
        p.Sta3n,
        p.DrugNameWithoutDose,
        p.DrugNameWithDose,
        'RxOut' AS SourceSystem,
        p.IssueDateTime AS MedicationDateTime,
        p.IssueDateTime AS StartDate,
        p.ExpirationDateTime AS EndDate,
        p.RxStatus AS Status,
        p.DaysSupply,
        p.Quantity,
        p.DEASchedule,
        p.ControlledSubstanceFlag,
        p.PrescriptionNumber AS OrderNumber,
        p.ProviderSID,
        p.LocalDrugSID,
        p.NationalDrugSID
    FROM RxOut.RxOutpat p
    WHERE p.IssueDateTime >= '{start_date}'
      AND p.IssueDateTime <= '{end_date}'
      AND p.RxStatus IN ('ACTIVE', 'DISCONTINUED', 'EXPIRED')
),
InpatientMeds AS (
    SELECT 
        m.PatientSID,
        m.PatientIEN,
        m.Sta3n,
        m.DrugNameWithoutDose,
        m.DrugNameWithDose,
        'BCMA' AS SourceSystem,
        m.ActionDateTime AS MedicationDateTime,
        m.ActionDateTime AS StartDate,
        NULL AS EndDate,
        m.ActionType AS Status,
        NULL AS DaysSupply,
        NULL AS Quantity,
        NULL AS DEASchedule,
        NULL AS ControlledSubstanceFlag,
        m.OrderNumber,
        m.OrderingProviderSID AS ProviderSID,
        m.LocalDrugSID,
        m.NationalDrugSID
    FROM BCMA.BCMAMedicationLog m
    WHERE m.ActionDateTime >= '{start_date}'
      AND m.ActionDateTime <= '{end_date}'
      AND m.ActionType IN ('{action_types_sql}')
)
SELECT * FROM OutpatientMeds
UNION ALL
SELECT * FROM InpatientMeds
ORDER BY PatientSID, MedicationDateTime;
"""

logging.info("SQL query defined")
logging.info(f"Query filters: Date range {start_date} to {end_date}")
logging.info(f"Query filters: RxOut status IN ('ACTIVE', 'DISCONTINUED', 'EXPIRED')")
logging.info(f"Query filters: BCMA action types IN {BCMA_INCLUDE_ACTION_TYPES}")

In [ ]:
# Execute query and load into DataFrame

logging.info("Executing SQL query...")
start_time = time.time()

df_medications = pd.read_sql(sql_query, conn)

elapsed = time.time() - start_time
logging.info(f"Successfully loaded {len(df_medications):,} rows, {len(df_medications.columns)} columns in {elapsed:.2f}s")

In [ ]:
# Close SQL Server connection

conn.close()
logging.info("SQL Server connection closed")

In [ ]:
# Take a look at DataFrame

df_medications.head(10)

In [ ]:
# Take a look at DataFrame (tail)

df_medications.tail(10)

In [ ]:
# Display DataFrame info

df_medications.info()

In [ ]:
# Display DataFrame shape and memory usage

print(f"Shape: {df_medications.shape}")
print(f"Rows: {len(df_medications):,}")
print(f"Columns: {len(df_medications.columns)}")
print(f"Memory usage: {df_medications.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Data quality checks

print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

# Check source system distribution
print("\nSource System Distribution:")
print(df_medications['SourceSystem'].value_counts())

# Check for null drug names
null_drug_names = df_medications['DrugNameWithDose'].isnull().sum()
print(f"\nNull DrugNameWithDose: {null_drug_names} ({null_drug_names/len(df_medications)*100:.2f}%)")

# Check unique patients
unique_patients = df_medications['PatientSID'].nunique()
print(f"\nUnique patients: {unique_patients}")

# Check date range
print(f"\nDate Range:")
print(f"  Earliest: {df_medications['MedicationDateTime'].min()}")
print(f"  Latest:   {df_medications['MedicationDateTime'].max()}")

print("=" * 60)

In [ ]:
# Create S3FileSystem for MinIO (pandas/pyarrow I/O)

logging.info(f"Initializing S3FileSystem for MinIO at {MINIO_ENDPOINT}")
fs = s3fs.S3FileSystem(
    anon=False,
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    client_kwargs={
        'endpoint_url': f"http://{MINIO_ENDPOINT}"
    }
)
logging.info("S3FileSystem created successfully")

In [ ]:
# Write unified medication DataFrame to v1_raw as Parquet

parquet_filename = "medications_combined.parquet"
parquet_uri = f"s3://{DEST_BUCKET}/{V1_RAW_MEDICATIONS_PREFIX}{parquet_filename}"
logging.info(f"Writing Parquet: {parquet_uri}")

start_time = time.time()

df_medications.to_parquet(
    parquet_uri,
    engine='pyarrow',
    filesystem=fs,
    compression='snappy',
    index=False
)

elapsed = time.time() - start_time
logging.info(f"Successfully wrote {len(df_medications):,} rows in {elapsed:.2f}s")

In [ ]:
# Verify write by reading back from v1_raw

logging.info("Verifying write by reading back from v1_raw...")

start_time = time.time()
df_verify = pd.read_parquet(parquet_uri, filesystem=fs)
elapsed = time.time() - start_time

logging.info(f"Verification: Read {len(df_verify):,} rows in {elapsed:.2f}s")

# Check row count matches
assert len(df_verify) == len(df_medications), f"Row count mismatch! Original: {len(df_medications)}, Read back: {len(df_verify)}"
logging.info("✓ Verification successful - row counts match")

# Check column count matches
assert len(df_verify.columns) == len(df_medications.columns), f"Column count mismatch!"
logging.info("✓ Verification successful - column counts match")

In [ ]:
# Display first few rows of verified data

df_verify.head()

In [ ]:
# Summary

print("\n" + "=" * 80)
print("DATA PREPARATION SUMMARY - PATIENT MEDICATIONS")
print("=" * 80)
print(f"Source:        SQL Server {SQLSERVER_SERVER}/{SQLSERVER_DATABASE}")
print(f"Schemas:       RxOut (outpatient) + BCMA (inpatient)")
print(f"Destination:   s3://{DEST_BUCKET}/{V1_RAW_MEDICATIONS_PREFIX}{parquet_filename}")
print(f"Date Range:    {start_date} to {end_date} ({(end_date - start_date).days} days)")
print(f"Rows:          {len(df_medications):,}")
print(f"Columns:       {len(df_medications.columns)}")
print(f"Unique Pts:    {df_medications['PatientSID'].nunique()}")
print(f"RxOut Meds:    {len(df_medications[df_medications['SourceSystem']=='RxOut']):,}")
print(f"BCMA Meds:     {len(df_medications[df_medications['SourceSystem']=='BCMA']):,}")
print(f"Status:        ✓ Complete")
print("=" * 80)
print("\nNext step: Run 02_explore.ipynb for exploratory data analysis")
print("Note: Can now join medications_combined.parquet with DDI reference data")